# Notebook 33 — Planning: CoT, ToT, GoT

Structured reasoning strategies beyond a single LLM call.

| Planner | Method |
|---|---|
| `ChainOfThought` | sequential step-by-step reasoning |
| `TreeOfThoughts` | BFS / DFS / beam search over branches |
| `GraphOfThoughts` | aggregate+fork — thoughts can merge |
| `StepBackPlanner` | abstract principle first, then apply |
| `ReActPlanner` | Reason + Act loop with tool calls |
| `PlanAndExecute` | decompose goal, then execute each step |

## 1. Setup — mock LLM

In [ ]:
from multigen.planning import (
    ChainOfThought, TreeOfThoughts, GraphOfThoughts,
    StepBackPlanner, PlanAndExecute, ReActPlanner,
    ThoughtNode, ExecutionPlan, PlanStep,
)

call_count = 0
async def mock_llm(prompt: str) -> str:
    global call_count
    call_count += 1
    # Return a structured response based on prompt type
    if 'rate the quality' in prompt.lower():
        return '0.8'
    if 'decompose' in prompt.lower() or 'numbered' in prompt.lower():
        return '1. Research the topic\n2. Outline key points\n3. Draft the content\n4. Review and refine'
    if 'finish[' in prompt.lower() or 'Action' in prompt:
        return 'Finish[The answer is 42]'
    return f'Thoughtful response about: {prompt[:50].replace(chr(10)," ")}'

print('Mock LLM ready')

## 2. Chain of Thought

In [ ]:
call_count = 0
cot = ChainOfThought(mock_llm, n_steps=3)
node = await cot.think('How do I optimise a slow database query?')

print(f'ChainOfThought result ({call_count} LLM calls):')
print(f'  depth: {node.depth}, terminal: {node.is_terminal}')
print(f'  steps: {len(node.metadata.get("steps", []))}')
print(f'  answer: {node.content[:80]}...')

## 3. Tree of Thoughts — beam search

In [ ]:
call_count = 0
tot = TreeOfThoughts(
    thinker_fn=mock_llm,
    branching_factor=3,
    max_depth=3,
    search='beam',
    beam_width=2,
)
best = await tot.think('Design a scalable notification system')
all_nodes = tot.all_nodes()

print(f'TreeOfThoughts beam search ({call_count} LLM calls):')
print(f'  Total nodes explored: {len(all_nodes)}')
print(f'  Best node score: {best.score:.2f}')
print(f'  Best node depth: {best.depth}')
print(f'  Answer: {best.content[:80]}...')

## 4. Tree of Thoughts — BFS vs DFS

In [ ]:
for strategy in ['bfs', 'dfs']:
    call_count = 0
    tot_s = TreeOfThoughts(mock_llm, branching_factor=2, max_depth=2, search=strategy)
    result = await tot_s.think('What is the best caching strategy?')
    print(f'{strategy.upper()}: {call_count} calls, {len(tot_s.all_nodes())} nodes, best_score={result.score:.2f}')

## 5. Graph of Thoughts

In [ ]:
call_count = 0
got = GraphOfThoughts(mock_llm, n_rounds=2, n_thoughts_per_round=2)
result = await got.think('Compare microservices vs monolith architecture')

print(f'GraphOfThoughts ({call_count} LLM calls):')
print(f'  Nodes: {len(got.all_nodes())}')
print(f'  Final thought: {result.content[:80]}...')

## 6. Step-Back Planner

In [ ]:
sbp = StepBackPlanner(mock_llm)
result = await sbp.think('Why does my React component re-render on every keystroke?')

print('StepBack result:')
print(f'  Abstract principle: {result.metadata["abstract"][:60]}...')
print(f'  Final answer: {result.content[:80]}...')

## 7. Plan and Execute

In [ ]:
pe = PlanAndExecute(mock_llm, max_steps=5)

# Phase 1: Plan
plan = await pe.plan('Write a comprehensive market analysis report on AI hardware')
print(f'Generated plan ({len(plan.steps)} steps):')
for i, step in enumerate(plan.steps, 1):
    print(f'  Step {i}: {step.description}')

# Phase 2: Execute
final = await pe.execute(plan)
print(f'\nExecution complete: {plan.is_done()}')
print(f'Final synthesis ({len(final)} chars):', final[:100], '...')

## 8. ReAct — Reason + Act

In [ ]:
search_results = {
    'Search[GDP France]': 'France GDP 2024: approximately $3.1 trillion USD',
    'Search[population Paris]': 'Paris population: approximately 2.16 million (city proper)',
}

async def dispatch(action: str) -> str:
    for key, val in search_results.items():
        if key.lower() in action.lower():
            return val
    return 'No result found for: ' + action

react = ReActPlanner(mock_llm, action_fn=dispatch, max_iterations=4)
result = await react.think('What is the GDP per capita of Paris, France?')

print('ReAct trajectory:')
for step in result.metadata['trajectory']:
    print(' ', step[:70])
print(f'\nFinal answer: {result.content[:80]}')


## Summary

| Planner | Best for | LLM calls |
|---|---|---|
| `ChainOfThought` | step-by-step reasoning | O(n_steps) |
| `TreeOfThoughts` | creative exploration, beam search | O(branch × depth) |
| `GraphOfThoughts` | multi-perspective synthesis | O(rounds × branches) |
| `StepBackPlanner` | physics/math/debugging | 3 |
| `PlanAndExecute` | long-horizon tasks | O(steps) |
| `ReActPlanner` | tool-use + reasoning | O(iterations × 2) |